# DTW deepfake classifier — scalable training

Trains the three-arm ablation (**baseline / dtw / fusion**) over however many LAV-DF
videos you have features for. It scales: point it at 60 videos or 6,000, the cells
do not change.

**What this notebook assumes you already have in Drive**, produced by
`lavdf_baseline.ipynb`:

* `plan-source-groups.json` — the manifest, with `label`, `partition`, `feature_path`, `video_path`
* `features/*.npz` — Whisper + DINOv2 features, one per video
* `media/{train,dev,test}/*.mp4` — the clips themselves

This notebook adds the parts that one does not do: DTW timing features, the
ablation, a learning curve, and a deployable checkpoint.

**Research rules it enforces, so you do not have to remember them**

* Train on **train**, select hyperparameters and the threshold on **validation**.
* Read **test** once, at the end. Never tune against it.
* Arms needing DTW features cannot score clips whose face track failed, so the
  headline comparison uses the **common subset** every arm can score.
* Clips with no timing features are **dropped and counted**, never imputed.

Runtime: CPU is fine. DTW caching is MediaPipe-bound at roughly 5–10 s per video,
so budget about 15 minutes per 100 videos. Training itself takes seconds.

## 1 · Setup

Set `REPO_COMMIT` to the revision you reviewed. Pinning it means this notebook
runs the same code next month.

In [ ]:
REPO_COMMIT = ""      # e.g. "f5a76c5"; leave "" to take the branch tip
BRANCH      = "claude/compassionate-cannon-f7frn0"
REPO        = "https://github.com/Pradeepbeleri/deepfake-detection-dtw.git"

import subprocess, sys, os, pathlib
if not pathlib.Path('/content/repo').exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO,'/content/repo'], check=True)
if REPO_COMMIT:
    subprocess.run(['git','-C','/content/repo','checkout','--detach',REPO_COMMIT], check=True)
print(subprocess.run(['git','-C','/content/repo','rev-parse','--short','HEAD'],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# Dependencies. Colab usually has torch already; these are the extras.
!pip -q install mediapipe==0.10.21 "numpy==1.26.4" 2>&1 | tail -2
!pip -q install -e /content/repo --no-deps 2>&1 | tail -2
import sys; sys.path.insert(0, '/content/repo/src')
import deepfake_dtw, numpy, torch
print('numpy', numpy.__version__, '| torch', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Point at your data

`PRIVATE` is the folder holding `features/`, `media/` and the manifest. Everything
written stays inside it — nothing private goes near the repository.

In [ ]:
PRIVATE  = '/content/drive/MyDrive/deepfake-private/lavdf'
MANIFEST = f'{PRIVATE}/plan-source-groups.json'

# Where this notebook writes. Change RUN to keep several experiments side by side.
RUN      = 'dtw-run-01'
OUT      = f'{PRIVATE}/{RUN}'
CACHE    = f'{OUT}/dtw-cache.json'
ABLATION = f'{OUT}/ablation.json'
import os; os.makedirs(OUT, exist_ok=True)
print('writing to', OUT)

### Paths inside the manifest must resolve here

If the manifest was written on another machine its absolute paths will not exist
in this runtime. This rewrites them onto the current `PRIVATE` folder and changes
nothing else — labels, partitions and provenance are copied verbatim.

In [ ]:
import json, pathlib
rows = json.loads(pathlib.Path(MANIFEST).read_text())
fixed, missing_f, missing_v = [], 0, 0
for r in rows:
    row = dict(r)
    row['feature_path'] = f"{PRIVATE}/features/{pathlib.Path(r['feature_path']).name}"
    src = pathlib.Path(r['video_path'])
    row['video_path'] = f"{PRIVATE}/media/{src.parent.name}/{src.name}"
    missing_f += not pathlib.Path(row['feature_path']).is_file()
    missing_v += not pathlib.Path(row['video_path']).is_file()
    fixed.append(row)

LOCAL_MANIFEST = f'{OUT}/manifest-local.json'
pathlib.Path(LOCAL_MANIFEST).write_text(json.dumps(fixed, indent=1))

from collections import Counter
for part in ('train','validation','test'):
    c = Counter(r['label'] for r in fixed if r['partition']==part)
    print(f"{part:11} {sum(c.values()):4}  real={c['real']:4} fake={c['fake']:4}")
print(f"\nmissing features {missing_f} | missing videos {missing_v}")
assert missing_f == 0 and missing_v == 0, "fix PRIVATE or re-run extraction before continuing"
assert any(r['partition']=='test' for r in fixed), "a held-out test partition is required"

## 3 · DTW timing features

Extracts 16 kHz RMS audio energy and MediaPipe lip aperture per clip, aligns them
with band-constrained DTW, and stores the 16-feature vector.

Scales to any number of videos. Resume-safe: it refuses to overwrite an existing
cache, so change `RUN` to recompute. Clips whose face track fails are recorded as
failures, not silently dropped.

In [ ]:
import time
t0 = time.time()
!cd /content/repo && OMP_NUM_THREADS=2 python -m deepfake_dtw.dtw_classifier cache \
    "{LOCAL_MANIFEST}" "{CACHE}" --source video
print(f'\nelapsed {time.time()-t0:.0f}s')

In [ ]:
cache = json.loads(pathlib.Path(CACHE).read_text())
print('computed', cache['computed'], '| failed', cache['failed'])
print(Counter(v['reason'] for v in cache['failures'].values()))

`--source features` is an alternative that needs no media: it compares
frame-to-frame novelty of the Whisper and DINOv2 sequences instead. Faster, and it
avoids the acoustic-energy-versus-mouth-geometry mismatch. Worth running as a
second experiment.

## 4 · The ablation

Three arms, one protocol, differing only in input. `--weight-decay 0.1` was
selected on validation for 60 training videos; with substantially more data,
re-tune it in section 6 rather than assuming it still holds.

In [ ]:
!cd /content/repo && OMP_NUM_THREADS=2 python -m deepfake_dtw.dtw_classifier compare \
    "{LOCAL_MANIFEST}" "{CACHE}" "{ABLATION}" \
    --hidden 128 --epochs 200 --weight-decay 0.1 --seed 42 --save-model fusion

In [ ]:
rep = json.loads(pathlib.Path(ABLATION).read_text())
print('common test videos:', rep['common_test_videos'],
      '| excluded:', rep['test_videos_excluded_from_common'])
print(f"\n{'arm':10}{'acc':>8}{'prec':>8}{'rec':>8}{'F1':>8}{'AUC':>9}{'FPR':>8}")
for m, a in rep['arms'].items():
    t = a['test_common_subset']
    print(f"{m:10}{t['accuracy']:8.4f}{t['precision']:8.4f}{t['recall']:8.4f}"
          f"{t['f1']:8.4f}{t['roc_auc']:9.4f}{t['false_positive_rate']:8.4f}")
    print(f"{'':10}confusion {t['confusion_matrix']}  (rows actual real/fake)")

### Is the difference real?

A single seed is not evidence. This repeats the comparison and reports the spread.
If the arms' ranges overlap, **say they are indistinguishable** — do not report the
mean difference as an improvement.

In [ ]:
import statistics as st
sys.path.insert(0, '/content/repo/src')
from deepfake_dtw.dataset import load
from deepfake_dtw.dtw_classifier import (load_cache, prepare_items, train_arm,
                                         evaluate_arm)

recs  = load(LOCAL_MANIFEST, feature_partitions={'train','validation','test'})
items = prepare_items(recs, require_video=True)
cobj  = load_cache(CACHE)
tr  = [i for i in items if i['partition']=='train']
va  = [i for i in items if i['partition']=='validation']
te  = [i for i in items if i['partition']=='test']
usable = set(cobj['entries'])
common = [i for i in te if i['sample_id'] in usable]
HP = dict(hidden=128, epochs=200, weight_decay=0.1)

scores = {}
for mode in ('baseline','dtw','fusion'):
    a = [evaluate_arm(train_arm(tr, va, cobj, mode, seed=s, **HP), common, cobj)
         for s in (1,2,3,4,5)]
    accs = [x['accuracy'] for x in a]; aucs = [x['roc_auc'] for x in a]
    scores[mode] = (accs, aucs)
    print(f"{mode:9} acc {st.mean(accs):.4f} [{min(accs):.3f},{max(accs):.3f}]   "
          f"AUC {st.mean(aucs):.4f} [{min(aucs):.3f},{max(aucs):.3f}]")

d = [f-b for f,b in zip(scores['fusion'][1], scores['baseline'][1])]
print(f"\nfusion - baseline AUC: mean {st.mean(d):+.4f}, "
      f"ahead in {sum(x>0 for x in d)}/5 seeds")
print("Seed spread is optimisation stability on one split, NOT sampling uncertainty.")

## 5 · Learning curve — does more data help?

Trains on stratified subsets of increasing size, scoring every point on the same
held-out videos. **If the curve is still rising at your largest size, collecting
more data is the highest-value next step.** If it has flattened, more of the same
data will not help and the bottleneck is elsewhere.

In [ ]:
import random
import matplotlib.pyplot as plt

real = [i for i in tr if i['label']==0]; fake = [i for i in tr if i['label']==1]
cap   = min(len(real), len(fake))
SIZES = [s for s in (16,24,32,40,48,60,80,120,200,400) if s//2 <= cap]
curve = {}
for size in SIZES:
    per = size//2; rows = {'baseline':[], 'fusion':[]}
    for seed in (1,2,3):
        rng = random.Random(1000+seed)
        sub = rng.sample(real, per) + rng.sample(fake, per)
        for mode in ('baseline','fusion'):
            rows[mode].append(evaluate_arm(
                train_arm(sub, va, cobj, mode, seed=seed, **HP), common, cobj))
    curve[size] = {m: (st.mean(x['accuracy'] for x in v),
                       st.mean(x['roc_auc'] for x in v)) for m,v in rows.items()}
    print(f"train={size:4}  baseline {curve[size]['baseline'][0]:.3f}/"
          f"{curve[size]['baseline'][1]:.3f}   fusion {curve[size]['fusion'][0]:.3f}/"
          f"{curve[size]['fusion'][1]:.3f}   (acc/AUC)")

fig, axes = plt.subplots(1, 2, figsize=(10,3.6))
for ax, idx, lab in ((axes[0],0,'Accuracy'), (axes[1],1,'ROC-AUC')):
    for mode, c, mk in (('baseline','#2a78d6','o'), ('fusion','#eb6834','^')):
        ax.plot(SIZES, [curve[s][mode][idx] for s in SIZES], color=c, marker=mk,
                linewidth=2, markersize=7, label=mode)
    ax.set_xlabel('Training videos'); ax.set_ylabel(lab); ax.legend(frameon=False)
    ax.grid(alpha=.3); ax.set_title(f'{lab} vs training-set size', loc='left')
fig.tight_layout(); fig.savefig(f'{OUT}/learning-curve.png', dpi=150)
plt.show()

## 6 · Re-tune when the dataset grows

The regularisation above suits 60 training videos. With more data the best setting
usually moves — more data tolerates (and needs) less regularisation.

**This cell reads validation only.** Never pick a configuration by test score.

In [ ]:
import itertools
best = None
for hidden, wd in itertools.product((32,128,256), (1e-3,1e-2,1e-1)):
    aucs = []
    for seed in (1,2,3):
        arm = train_arm(tr, va, cobj, 'fusion', hidden=hidden, epochs=200,
                        weight_decay=wd, seed=seed)
        aucs.append(arm['validation_metrics']['roc_auc'])
    m = st.mean(aucs)
    print(f'  hidden {hidden:4}  wd {wd:6}  validation AUC {m:.4f}')
    if best is None or m > best[0]:
        best = (m, hidden, wd)
print(f'\nbest on validation: hidden {best[1]}, weight decay {best[2]} (AUC {best[0]:.4f})')
print('Re-run section 4 with these, then read test ONCE.')

## 7 · Deploy the checkpoint

`compare --save-model fusion` already wrote `ablation.model.pt`. The service needs
its SHA-256, which is also its integrity check — the loader refuses a file whose
hash does not match.

In [ ]:
import hashlib
model = ABLATION.replace('.json', '.model.pt')
digest = hashlib.sha256(pathlib.Path(model).read_bytes()).hexdigest()
print('checkpoint :', model)
print('sha256     :', digest)
print(f'''
Set these where the local service runs:

  export DEEPFAKE_TEMPORAL_CHECKPOINT="/your/private/path/{pathlib.Path(model).name}"
  export DEEPFAKE_TEMPORAL_SHA256="{digest}"

Keep the checkpoint outside Git. It is derived from LAV-DF, so the provider's
non-commercial research terms and the underlying VoxCeleb2 conditions apply to it.''')

## What to claim, and what not to

* **Fair to say:** measured accuracy and ROC-AUC on a held-out partition read once,
  with the seed spread and the test size stated.
* **Not fair to say:** that DTW improves detection, unless the arms' ranges separate.
  Overlapping ranges mean *indistinguishable*, whatever the means are.
* **Still unverified:** identity-disjointness. Source-group splitting does not
  guarantee the same speaker is absent from two partitions, so part of any score
  may be speaker recognition.
* **Scores are uncalibrated.** They are not probabilities.
* **One dataset, one manipulation family.** Nothing here transfers to a generator
  the model never saw.